# Streaming Drift + Anomaly Detection (Isolation Forest)

Portfolio-grade notebook (CPU-friendly) that demonstrates:
- synthetic telemetry generation (normal + drift + injected anomalies)
- **streaming evaluation** (no lookahead)
- Isolation Forest scoring
- threshold calibration with a held-out *clean* baseline window
- alert reporting

Outputs are saved inside the notebook when executed.

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_recall_fscore_support

SEED = 1337
rng = np.random.default_rng(SEED)
pd.set_option('display.max_columns', 50)

## 1) Generate synthetic stream
We create minute-level aggregates per host: connections, bytes, entropy.
Then we introduce drift (baseline mean changes) and sparse anomalies.

In [2]:
n = 8000
t = np.arange(n)

# baseline
connections = rng.poisson(lam=10, size=n).astype(float)
bytes_ = np.clip(rng.normal(120_000, 35_000, size=n), 1000, None)
entropy = np.clip(rng.normal(3.6, 0.5, size=n), 0.0, 8.0)

# drift: after 60% of the stream, increase typical connections + bytes
drift_start = int(n * 0.6)
connections[drift_start:] += rng.poisson(lam=3, size=n - drift_start)
bytes_[drift_start:] *= rng.normal(1.15, 0.05, size=n - drift_start)
entropy[drift_start:] += rng.normal(0.15, 0.05, size=n - drift_start)

# anomalies: sparse spikes
y = np.zeros(n, dtype=int)
anom_idx = rng.choice(np.arange(200, n), size=35, replace=False)
y[anom_idx] = 1
connections[anom_idx] += rng.integers(80, 200, size=len(anom_idx))
bytes_[anom_idx] *= rng.uniform(2.0, 6.0, size=len(anom_idx))
entropy[anom_idx] = np.clip(entropy[anom_idx] + rng.uniform(1.5, 3.0, size=len(anom_idx)), 0, 8)

df = pd.DataFrame({
    't': t,
    'connections': connections,
    'bytes': bytes_,
    'entropy': entropy,
    'is_anom': y,
})
df.head(), df['is_anom'].mean()

(   t  connections          bytes   entropy  is_anom
 0  0         14.0  137602.010232  4.274441        0
 1  1         14.0   82978.939658  3.360939        0
 2  2          7.0  150179.831970  3.864977        0
 3  3         10.0  128704.209037  3.909155        0
 4  4         13.0  142578.729565  3.642974        0,
 np.float64(0.004375))

## 2) Streaming features (rolling stats)
We compute rolling z-scores using only past data.

In [3]:
window = 120
feat = df.copy()
for col in ['connections','bytes','entropy']:
    mu = feat[col].rolling(window, min_periods=20).mean().shift(1)
    sd = feat[col].rolling(window, min_periods=20).std(ddof=0).shift(1).fillna(0.0)
    feat[f'z_{col}'] = ((feat[col] - mu) / (sd + 1e-8)).abs().fillna(0.0)

X = feat[[f'z_{c}' for c in ['connections','bytes','entropy']]].values
X[:3]

array([[0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.]])

## 3) Train Isolation Forest on a clean baseline window
We calibrate threshold on that baseline to target a low alert rate.

In [4]:
baseline_end = 2500
baseline_mask = (feat['t'] < baseline_end) & (feat['is_anom'] == 0)
X_base = X[baseline_mask.values]

iso = IsolationForest(
    n_estimators=300,
    contamination='auto',
    random_state=SEED,
    n_jobs=-1,
)
iso.fit(X_base)

# higher score = more normal in sklearn; we invert to get anomaly score
score = -iso.score_samples(X)

# calibrate threshold on baseline: keep only top 0.3% as alerts
target_alert_rate = 0.003
thr = np.quantile(score[baseline_mask.values], 1 - target_alert_rate)
thr

np.float64(0.6957453808659391)

## 4) Alerts + evaluation (post-hoc)

In [5]:
feat['anomaly_score'] = score
feat['alert'] = (feat['anomaly_score'] >= thr).astype(int)

precision, recall, f1, _ = precision_recall_fscore_support(
    feat['is_anom'], feat['alert'], average='binary', zero_division=0
)
precision, recall, f1

(0.5737704918032787, 1.0, 0.7291666666666666)

In [6]:
alerts = feat.query('alert==1').sort_values('anomaly_score', ascending=False).head(15)
alerts[['t','connections','bytes','entropy','anomaly_score','is_anom']].head(15)

,t,connections,bytes,entropy,anomaly_score,is_anom
306,306,103.0,821597.150698,5.808101,0.805508,1
209,209,185.0,516995.303636,6.213319,0.805508,1
1270,1270,174.0,596577.223100,6.553347,0.805508,1
925,925,191.0,409107.990878,5.731402,0.805508,1
687,687,175.0,400794.395399,5.561077,0.805508,1
1003,1003,130.0,467415.913276,6.705004,0.805508,1
1930,1930,175.0,706513.998236,6.895388,0.805508,1
1684,1684,162.0,330557.726461,6.502210,0.805508,1
2513,2513,162.0,587736.582031,6.918337,0.805508,1
1814,1814,149.0,416827.927912,5.462465,0.805508,1


## 5) Drift note
This baseline-trained model will typically fire more alerts after drift starts.
Next step: periodic recalibration, adaptive thresholds, or concept drift detectors.